# 01 · Data Preprocessing
**Cross-Platform Social Media Trend Intelligence & Prediction System**

This notebook covers: dataset loading, inspection, missing-value handling,
duplicate removal, data-type conversion, outlier handling, feature
engineering (Engagement Rate, Trend Score, Trend Label), categorical
encoding, numerical scaling, and the train/test split.

All logic here mirrors `utils/preprocessing.py`, which the Flask app
imports directly so the notebook and the web app never drift apart.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
from utils.preprocessing import (
    load_raw_dataset, clean_dataset, add_engagement_rate, add_trend_score,
    prepare_model_data, MODEL_FEATURES, TARGET
)

pd.set_option('display.max_columns', None)
DATASET_PATH = os.path.join('..', 'dataset', 'social_media_trends.csv')

## 1. Dataset Loading & Inspection

In [2]:
raw_df = load_raw_dataset(DATASET_PATH)
print("Shape:", raw_df.shape)
raw_df.head()

Shape: (8040, 12)


,post_id,platform,date,topic,category,views,likes,comments,shares,growth_rate,sentiment,sentiment_score
0,P102835,X,2024-10-16,Home Decor,Lifestyle,6466,496,52,59,5.06,Negative,-0.204
1,P100737,YouTube,2025-01-30,Celebrity News,Entertainment,19041,890,71,147,0.40,Negative,-0.220
2,P103334,Instagram,2025-01-17,Patch Update,Gaming,10553,257,50,53,1.86,Negative,-0.891
3,P107386,TikTok,2024-01-13,Home Decor,Lifestyle,31080,1178,201,262,2.76,Negative,-0.651
4,P101373,YouTube,2025-06-27,Athlete Update,Sports,28751,703,119,170,0.42,Negative,-1.000


In [3]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8040 entries, 0 to 8039
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   post_id          8040 non-null   str    
 1   platform         8040 non-null   str    
 2   date             8040 non-null   str    
 3   topic            8040 non-null   str    
 4   category         8040 non-null   str    
 5   views            8040 non-null   int64  
 6   likes            8040 non-null   int64  
 7   comments         8040 non-null   int64  
 8   shares           8040 non-null   int64  
 9   growth_rate      8010 non-null   float64
 10  sentiment        8040 non-null   str    
 11  sentiment_score  7980 non-null   float64
dtypes: float64(2), int64(4), str(6)
memory usage: 753.9 KB


In [4]:
raw_df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
post_id,8040,8000,P100706,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
platform,8040,4,Instagram,2058,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8040,730,2024-08-16,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
topic,8040,40,Designer Launch,226,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,8040,8,News,1047,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,8040.0,NaN,NaN,NaN,24774.699627,53758.303614,1888.0,11758.0,18381.5,29963.25,3273340.0
likes,8040.0,NaN,NaN,NaN,1657.120398,2030.819818,13.0,497.0,977.0,2001.0,31507.0
comments,8040.0,NaN,NaN,NaN,220.599502,294.181706,3.0,62.75,126.0,261.0,6552.0
shares,8040.0,NaN,NaN,NaN,309.450871,401.201219,4.0,88.0,178.0,370.0,8542.0
growth_rate,8010.0,NaN,NaN,NaN,10.611336,10.614577,-20.0,3.03,10.09,17.71,51.23


## 2. Missing Values

In [5]:
print("Missing values per column:")
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0])

Missing values per column:
growth_rate        30
sentiment_score    60
dtype: int64


We impute `sentiment_score` and `growth_rate` with their **median**
(robust to outliers) rather than the mean, and drop any row still missing
a critical identifying field (`post_id`, `platform`, `category`, `views`).

## 3. Duplicate Checking

In [6]:
dupe_count = raw_df.duplicated(subset=['post_id']).sum()
print(f"Duplicate post_id rows found: {dupe_count}")

Duplicate post_id rows found: 40


## 4. Data Cleaning (missing values, duplicates, dtypes, outliers)

In [7]:
clean_df = clean_dataset(raw_df)
print("Shape after cleaning:", clean_df.shape)
print("Duplicates removed:", clean_df.attrs.get('removed_duplicates'))
print("\nRemaining missing values:\n", clean_df.isnull().sum().sum())

Shape after cleaning: (8000, 12)
Duplicates removed: 40

Remaining missing values:
 0


**Outlier handling:** `views` is capped at the 99th percentile rather
than dropped, so a handful of viral posts don't dominate scaling/model
training, while still keeping the record in the dataset.

## 5. Feature Engineering

### Engagement Rate
$$\text{Engagement Rate} = \frac{Likes + Comments + Shares}{Views} \times 100$$
Division by zero is handled safely (views of 0 would produce NaN, which is
filled with 0).

In [8]:
fe_df = add_engagement_rate(clean_df)
fe_df[['views','likes','comments','shares','engagement_rate']].head()

,views,likes,comments,shares,engagement_rate
0,6466.0,496,52,59,9.387566
1,19041.0,890,71,147,5.819022
2,10553.0,257,50,53,3.411352
3,31080.0,1178,201,262,5.279923
4,28751.0,703,119,170,3.450315


### Project-Defined Trend Score
This is **not** an official social-media metric — it is defined for this
academic project as a weighted combination of engagement, growth and reach:

$$\text{Trend Score} = 0.4 \times \text{norm(engagement\_rate)} + 0.3 \times \text{norm(growth\_rate)} + 0.3 \times \text{norm(views)}$$

Each component is **percentile-rank normalized** to [0,1] before combining
(rather than plain min-max), because raw views/engagement are right-skewed
with a long viral tail — min-max normalization let a handful of outliers
compress nearly every other post toward 0, producing a badly imbalanced
Low/Moderate/High split. Percentile-rank normalization is robust to that
skew and produces a well-balanced trend distribution.

Thresholds: Low `< 40`, Moderate `40–70`, High `>= 70`.

In [9]:
processed_df = add_trend_score(fe_df)
processed_df[['engagement_rate','growth_rate','views','trend_score','trend_label']].head()

,engagement_rate,growth_rate,views,trend_score,trend_label
0,9.387566,5.06,6466.0,38.43,Low
1,5.819022,0.40,19041.0,34.61,Low
2,3.411352,1.86,10553.0,16.24,Low
3,5.279923,2.76,31080.0,41.58,Moderate
4,3.450315,0.42,28751.0,31.13,Low


In [10]:
processed_df['trend_label'].value_counts()

trend_label
Moderate    3086
Low         2995
High        1919
Name: count, dtype: int64

## 6. Categorical Encoding, Feature Scaling & Train/Test Split

Encoding and scaling are fit **only on the training split** to avoid
data leakage. The target (`trend_label`) is excluded from the input
features, and `trend_score` itself is also excluded from `MODEL_FEATURES`
since it's directly derived from `engagement_rate`, `growth_rate` and
`views` — including it would leak the answer.

In [11]:
print("Model input features:", MODEL_FEATURES)
print("Target:", TARGET)

X_train, X_test, y_train, y_test, transformers = prepare_model_data(
    processed_df, save_dir=os.path.join('..', 'models')
)
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Class balance (train):", np.bincount(y_train))
print("Class balance (test):", np.bincount(y_test))

Model input features: ['platform', 'category', 'views', 'likes', 'comments', 'shares', 'growth_rate', 'sentiment_score', 'engagement_rate']
Target: trend_label

X_train shape: (6400, 9)
X_test shape: (1600, 9)
Class balance (train): [2396 2469 1535]
Class balance (test): [599 617 384]


In [12]:
X_train.head()

,platform,category,views,likes,comments,shares,growth_rate,sentiment_score,engagement_rate
3147,2,4,-0.063073,-0.073332,-0.131927,-0.081915,-0.295610,0.878138,0.198688
1740,2,4,-0.827966,-0.630973,-0.554765,-0.535668,-0.083188,-0.193178,-0.533166
1160,2,0,-1.215443,-0.756504,-0.739535,-0.686069,1.021599,0.529960,-0.407450
2012,0,6,-0.201084,-0.119843,-0.426848,-0.084464,0.084664,0.332741,0.230769
4350,0,0,-0.555041,-0.618969,-0.647150,-0.607045,-0.821925,-2.065547,-1.047719


## 7. Save Processed Dataset
The cleaned + feature-engineered (but still human-readable) dataframe is
also saved so the EDA notebook can load it directly.

In [13]:
processed_df.to_csv(os.path.join('..', 'dataset', 'processed_social_media_trends.csv'), index=False)
print("Saved processed_social_media_trends.csv")

Saved processed_social_media_trends.csv


## Summary
- Loaded raw dataset, inspected structure and missing values
- Cleaned duplicates, fixed data types, capped extreme outliers
- Engineered `engagement_rate` and the project-defined `trend_score` / `trend_label`
- Encoded categoricals, scaled numeric features, split into train/test with **no data leakage**
- Saved `preprocessing.pkl` transformers (reused identically by the Flask app) and the processed dataset for EDA